In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
from typing import List, Any
import sys

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

from src.data_loader import DataLoader
from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer

print("✅ All imports successful")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")


✅ All imports successful
   pandas  : 2.3.3
   numpy   : 1.26.4


In [2]:
# Load Raw Data
loader = DataLoader("../config/config.yaml")
df_raw = loader.load_data()

print(f"\n📊 Raw Data Shape : {df_raw.shape}")
print(f"   Total Missing  : {df_raw.isnull().sum().sum()}")
print(f"   Dtypes         :")
print(df_raw.dtypes.value_counts())

df_raw.head(3)

INFO:src.data_loader:✅ Config loaded from: ../config/config.yaml
INFO:src.data_loader:✅ DataLoader initialized successfully.
INFO:src.data_loader:🔄 Loading data from: d:\STUDY\Programming\project\house-price-prediction\data\raw\housing_data.csv
INFO:src.data_loader:✅ Data loaded successfully. Shape: (90000, 28)



📊 Raw Data Shape : (90000, 28)
   Total Missing  : 5420
   Dtypes         :
int64     14
object    14
Name: count, dtype: int64


,LotArea,YearBuilt,YearRemodAdd,TotalBsmtSF,GrLivArea,FullBath,HalfBath,BedroomAbvGr,TotRmsAbvGrd,GarageArea,...,Neighborhood,BldgType,HouseStyle,RoofStyle,ExterQual,HeatingQC,KitchenQual,GarageType,SaleCondition,SalePrice
0,11051,1901,1901,982,1447,0,0,2,4,466,...,NAmes,1Fam,1Story,Gable,Gd,Ex,TA,BuiltIn,Partial,233400
1,6930,1918,1918,648,1449,2,0,3,9,536,...,OldTown,1Fam,2Story,Gable,Gd,Gd,TA,Detchd,Partial,312900
2,7588,1906,2009,0,1376,0,0,3,7,0,...,OldTown,1Fam,2Story,Hip,TA,TA,TA,NaN,Normal,210800


In [3]:
# Missing Value Deep Dive
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    "Missing Count"  : missing,
    "Missing %"      : missing_pct,
    "Dtype"          : df_raw.dtypes,
}).query("`Missing Count` > 0").sort_values("Missing %", ascending=False)

print("=" * 50)
print("     MISSING VALUE SUMMARY")
print("=" * 50)
print(missing_df)

     MISSING VALUE SUMMARY
            Missing Count  Missing %   Dtype
GarageType           5420       6.02  object


In [4]:
# Plot missing values
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#EF4444" if p > 50 else "#F59E0B" if p > 20 else "#2563EB"
              for p in missing_df["Missing %"]]
    bars = ax.bar(missing_df.index, missing_df["Missing %"], color=colors)
    ax.set_xlabel("Feature")
    ax.set_ylabel("Missing %")
    ax.set_title("Missing Values by Feature", fontsize=13, fontweight="bold")
    ax.axhline(y=50, color="red", linestyle="--", alpha=0.5, label="50% threshold")
    for bar, val in zip(bars, missing_df["Missing %"]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.5,
                f"{val}%", ha="center", fontsize=9)
    plt.xticks(rotation=45, ha="right")
    plt.legend()
    plt.tight_layout()
    plt.savefig("../reports/figures/missing_values.png", dpi=150)
    plt.show()
else:
    print("✅ No missing values found!")

In [5]:
# Before vs After — Missing Value Imputation 
preprocessor = DataPreprocessor("../config/config.yaml")
df_imputed = preprocessor.handle_missing_values(df_raw.copy())

print(f"Before imputation — Missing : {df_raw.isnull().sum().sum()}")
print(f"After  imputation — Missing : {df_imputed.isnull().sum().sum()}")

# Visual comparison for key numerical columns
num_cols = ["GrLivArea", "TotalBsmtSF", "GarageArea"]
fig, axes = plt.subplots(2, len(num_cols), figsize=(14, 8))
fig.suptitle("Distribution Before vs After Imputation",
             fontsize=13, fontweight="bold")

for i, col in enumerate(num_cols):
    if col in df_raw.columns:
        # Before
        axes[0, i].hist(df_raw[col].dropna(), bins=30,
                        color="#EF4444", alpha=0.7, edgecolor="white")
        axes[0, i].set_title(f"{col}\n(Before)")
        axes[0, i].set_ylabel("Frequency" if i == 0 else "")

        # After
        axes[1, i].hist(df_imputed[col].dropna(), bins=30,
                        color="#10B981", alpha=0.7, edgecolor="white")
        axes[1, i].set_title(f"{col}\n(After)")
        axes[1, i].set_ylabel("Frequency" if i == 0 else "")

plt.tight_layout()
plt.savefig("../reports/figures/imputation_comparison.png", dpi=150)
plt.show()

INFO:src.preprocessing:✅ DataPreprocessor initialized.
INFO:src.preprocessing:🔄 Handling missing values...
INFO:src.preprocessing:  Found missing values:
GarageType    5420
dtype: int64
INFO:src.preprocessing:✅ Missing values handled. Remaining: 0


Before imputation — Missing : 5420
After  imputation — Missing : 0


In [6]:
# Outlier Detection & Visualization
num_features = [
    "LotArea", "GrLivArea", "TotalBsmtSF",
    "GarageArea", "SalePrice"
]
num_features = [f for f in num_features if f in df_imputed.columns]

fig, axes = plt.subplots(2, len(num_features), figsize=(16, 8))
fig.suptitle("Outlier Analysis — Box Plots & Histograms",
             fontsize=13, fontweight="bold")

for i, col in enumerate(num_features):
    # Box plot
    axes[0, i].boxplot(df_imputed[col].dropna(), patch_artist=True,
                       boxprops=dict(facecolor="#2563EB", alpha=0.5))
    axes[0, i].set_title(col, fontsize=9)
    axes[0, i].set_ylabel("Value" if i == 0 else "")

    # Histogram
    axes[1, i].hist(df_imputed[col].dropna(), bins=30,
                    color="#2563EB", alpha=0.7, edgecolor="white")
    axes[1, i].set_xlabel(col, fontsize=9)
    axes[1, i].set_ylabel("Count" if i == 0 else "")

plt.tight_layout()
plt.savefig("../reports/figures/outlier_analysis.png", dpi=150)
plt.show()

In [7]:
# Outlier Removal Results
df_no_outliers = preprocessor.remove_outliers(df_imputed, method="iqr")

print(f"\n📊 Outlier Removal Summary:")
print(f"   Before : {len(df_imputed):,} rows")
print(f"   After  : {len(df_no_outliers):,} rows")
print(f"   Removed: {len(df_imputed) - len(df_no_outliers):,} rows "
      f"({(len(df_imputed)-len(df_no_outliers))/len(df_imputed)*100:.1f}%)")

INFO:src.preprocessing:🔄 Removing outliers using IQR (threshold=3.0)...
INFO:src.preprocessing:✅ Removed 5841 outlier rows (6.5%). Remaining: 84,159



📊 Outlier Removal Summary:
   Before : 90,000 rows
   After  : 84,159 rows
   Removed: 5,841 rows (6.5%)


In [8]:
# Categorical Encoding 
cat_cols = df_no_outliers.select_dtypes(include=["object"]).columns.tolist()
print(f"\nCategorical columns BEFORE encoding: {len(cat_cols)}")
print(f"Columns: {cat_cols}")

df_encoded = preprocessor.encode_categoricals(df_no_outliers)

cat_cols_after = df_encoded.select_dtypes(include=["object"]).columns.tolist()
print(f"\nCategorical columns AFTER encoding : {len(cat_cols_after)}")
print(f"Total columns after encoding       : {df_encoded.shape[1]}")

# Value counts for key categoricals
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cat_to_plot = ["MSZoning", "Neighborhood", "BldgType"]
cat_to_plot = [c for c in cat_to_plot if c in df_no_outliers.columns]

for i, col in enumerate(cat_to_plot):
    top_vals = df_no_outliers[col].value_counts().head(8)
    axes[i].bar(top_vals.index, top_vals.values,
                color=plt.cm.Blues(np.linspace(0.4, 0.9, len(top_vals))))
    axes[i].set_title(f"{col} Distribution")
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylabel("Count" if i == 0 else "")

plt.tight_layout()
plt.savefig("../reports/figures/categorical_distributions.png", dpi=150)
plt.show()

INFO:src.preprocessing:🔄 Encoding categorical features...
INFO:src.preprocessing:✅ Categorical encoding complete. Shape: (84159, 78)



Categorical columns BEFORE encoding: 14
Columns: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'Neighborhood', 'BldgType', 'HouseStyle', 'RoofStyle', 'ExterQual', 'HeatingQC', 'KitchenQual', 'GarageType', 'SaleCondition']

Categorical columns AFTER encoding : 0
Total columns after encoding       : 78


In [9]:
# Feature Engineering 
X_raw = df_no_outliers.drop(columns=["SalePrice"])
y_raw = df_no_outliers["SalePrice"]

engineer = FeatureEngineer()
X_engineered = engineer.run_all(X_raw, y_raw)

print(f"\n📊 Feature Engineering Summary:")
print(f"   Original features  : {X_raw.shape[1]}")
print(f"   Engineered features: {X_engineered.shape[1]}")
print(f"   New features added : {X_engineered.shape[1] - X_raw.shape[1]}")

new_features = [c for c in X_engineered.columns if c not in X_raw.columns]
print(f"\nNew features created:\n  {new_features}")

INFO:src.feature_engineering:✅ FeatureEngineer initialized.
INFO:src.feature_engineering:==================================================
INFO:src.feature_engineering:🚀 RUNNING FEATURE ENGINEERING PIPELINE
INFO:src.feature_engineering:==================================================
INFO:src.feature_engineering:🔄 Adding age features...
INFO:src.feature_engineering:✅ Age features added.
INFO:src.feature_engineering:🔄 Adding area features...
INFO:src.feature_engineering:✅ Area features added.
INFO:src.feature_engineering:🔄 Adding quality features...
INFO:src.feature_engineering:✅ Quality features added.
INFO:src.feature_engineering:🔄 Adding neighborhood features...
INFO:src.feature_engineering:✅ Neighborhood features added.
INFO:src.feature_engineering:🔄 Applying log transformation...
INFO:src.feature_engineering:  Auto-detected skewed columns (13): ['LotArea', 'YearRemodAdd', 'GrLivArea', 'HalfBath', 'PoolArea', 'RemodelAge', 'YearsAfterRemodel', 'HasPool', 'LotAreaPerRoom', 'AreaPe


📊 Feature Engineering Summary:
   Original features  : 27
   Engineered features: 54
   New features added : 27

New features created:
  ['HouseAge', 'RemodelAge', 'IsRemodeled', 'YearsAfterRemodel', 'TotalSF', 'TotalBathrooms', 'HasPool', 'HasGarage', 'LotAreaPerRoom', 'AreaPerBath', 'QualityScore', 'QualCondRatio', 'IsHighQuality', 'NeighborhoodAvgPrice', 'LotArea_log', 'YearRemodAdd_log', 'GrLivArea_log', 'HalfBath_log', 'PoolArea_log', 'RemodelAge_log', 'YearsAfterRemodel_log', 'HasPool_log', 'LotAreaPerRoom_log', 'AreaPerBath_log', 'QualCondRatio_log', 'IsHighQuality_log', 'NeighborhoodAvgPrice_log']


In [10]:
# Feature Scaling 
X_encoded = preprocessor.encode_categoricals(X_engineered)
X_train, X_test, y_train, y_test = preprocessor.split_data(X_encoded, y_raw)
X_train_scaled, X_test_scaled = preprocessor.scale_features(X_train, X_test)

print(f"\n📊 Final Dataset Summary:")
print(f"   X_train shape : {X_train_scaled.shape}")
print(f"   X_test  shape : {X_test_scaled.shape}")
print(f"   y_train range : ${y_train.min():,} — ${y_train.max():,}")
print(f"   y_test  range : ${y_test.min():,} — ${y_test.max():,}")

# Compare distributions before/after scaling
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cols_to_compare = ["GrLivArea", "LotArea"]

for col in cols_to_compare:
    if col in X_train.columns:
        axes[0].hist(X_train[col], bins=30, alpha=0.5, label=col)
        axes[1].hist(X_train_scaled[col], bins=30, alpha=0.5, label=col)

axes[0].set_title("Before Scaling")
axes[1].set_title("After RobustScaler")
for ax in axes:
    ax.legend()
    ax.set_ylabel("Frequency")

plt.suptitle("Feature Scaling Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/scaling_comparison.png", dpi=150)
plt.show()

INFO:src.preprocessing:🔄 Encoding categorical features...
INFO:src.preprocessing:✅ Categorical encoding complete. Shape: (84159, 104)
INFO:src.preprocessing:🔄 Splitting data (test_size=0.2, random_state=42)...
INFO:src.preprocessing:✅ Split complete:
   Training  : 67,327 samples
   Testing   : 16,832 samples
INFO:src.preprocessing:🔄 Scaling features...
INFO:src.preprocessing:✅ Feature scaling complete.



📊 Final Dataset Summary:
   X_train shape : (67327, 104)
   X_test  shape : (16832, 104)
   y_train range : $67,500 — $755,000
   y_test  range : $72,500 — $755,000


In [11]:
import os
os.makedirs("../data/processed", exist_ok=True)

train_df = X_train_scaled.copy()
train_df["SalePrice"] = y_train.values
train_df.to_csv("../data/processed/train_data.csv", index=False)

test_df = X_test_scaled.copy()
test_df["SalePrice"] = y_test.values
test_df.to_csv("../data/processed/test_data.csv", index=False)

print("✅ Processed data saved!")
print(f"   Train: ../data/processed/train_data.csv")
print(f"   Test : ../data/processed/test_data.csv")

✅ Processed data saved!
   Train: ../data/processed/train_data.csv
   Test : ../data/processed/test_data.csv
